# Wrong-way detection on Colab

Runs `pipeline.py` (RF-DETR + ByteTrack + `WrongWayDetector`) against a
dashcam video on a Colab GPU.

**Before running anything:** Runtime -> Change runtime type -> **T4 GPU**.
On a CPU runtime this works but is roughly an order of magnitude slower.

The first thing to read in the output of the run cell is the **class-id
table**, not the alerts. It is there to settle the open question in
`CLAUDE.md`: whether `VEHICLE_CLASS_IDS = {2, 3, 5, 7}` actually matches
this RF-DETR build. If the names next to those ids are not vehicles,
every alert below them is meaningless.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")

## Install

Only two packages. Colab already ships `torch` (with CUDA), `opencv` and
`numpy` -- reinstalling them from `requirements.txt` can swap the CUDA
build for a CPU one and quietly cost you the GPU. `supervision` arrives
as a dependency of `rfdetr`.

In [ ]:
!pip install -q rfdetr trackers

## Get the code and a video into the session

Upload `wrong_way_detector.py`, `pipeline.py` and one dashcam video.
Colab wipes the filesystem when the runtime disconnects, so this cell
runs again every session.

The repo is private, so `git clone` would need a token. If you ever make
it public, this whole cell collapses into:

```
!git clone https://github.com/Arielevi15/Crime_Traffic_Dedector.git
%cd Crime_Traffic_Dedector
```

In [ ]:
from google.colab import files

uploaded = files.upload()
print(sorted(uploaded))

## Run

`limit_frames=300` keeps the first run short -- long enough to produce
the class-id table and see whether tracking holds, short enough that a
misconfiguration costs seconds instead of an hour. Drop it once the
class ids are confirmed.

Set `VIDEO` to whatever you actually uploaded.

In [ ]:
from pipeline import run

VIDEO = "dashcam.mp4"

alerts = run(
    video=VIDEO,
    output="check.mp4",
    limit_frames=300,
)
alerts

## Watch the annotated result

Green box = tracked vehicle, red = alerted, orange dot = the road-contact
point the detector actually reasons about. If those dots are not landing
on the road under each vehicle, fix that before tuning any threshold.

OpenCV writes `mp4v`, which Colab's player will not decode, so re-encode
to H.264 first. The video is inlined as base64, which is fine for a few
hundred frames -- for a full clip use `files.download` in the next cell.

In [ ]:
from base64 import b64encode

from IPython.display import HTML

!ffmpeg -loglevel error -i check.mp4 -vcodec libx264 -y check_h264.mp4

payload = b64encode(open("check_h264.mp4", "rb").read()).decode()
HTML('<video width=720 controls><source src="data:video/mp4;base64,{0}">'.format(payload))

In [ ]:
# For anything longer than a short clip, download instead of inlining.
from google.colab import files

files.download("check_h264.mp4")

## Tuning

Once the class ids are confirmed, the next open task is tuning
`DetectorConfig` against real footage. Pass one in explicitly rather than
editing the module, so the defaults stay the tested ones:

```python
from wrong_way_detector import DetectorConfig

alerts = run(
    video=VIDEO,
    output="check.mp4",
    config=DetectorConfig(opposite_cos_threshold=-0.6),
)
```

Per `CLAUDE.md` principle 3, tune toward silence: a false positive
accuses an innocent driver, a false negative just misses one.